# 彩色装箱问题 (CBPP)

**类别:** 装箱

来源: [https://www.hexaly.com/templates/colored-bin-packing-problem-cbpp](https://www.hexaly.com/templates/colored-bin-packing-problem-cbpp)


## 问题描述

**在彩色装箱问题 (BPP)** 中,一组具有已知重量和颜色的物品必须分配到具有相同容量的箱子中。每个物品必须被放入恰好一个箱子内。每个箱子内物品的总重量不能超过其容量,且箱子内的所有物品必须具有不同的颜色。目标是最小化所用箱子的数量。该问题是 [装箱问题 (BPP)](https://www.hexaly.com/docs/last/exampletour/binpacking.html) 的一个变种,因此是 NP 难的。

	

### 学到的要点

- 添加 [集合决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模箱子的内容
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算箱子的总重量以及箱子中给定颜色的元素数量


## 数据

所提供的彩色装箱问题 (CBPP) 实例来自 AI-Q2 数据集,它是 [BPPLIB](http://or.dei.unibo.it/library/bpplib) 中 Augmented IRUP (AI) 实例的一个改编版本,其中每个物品的颜色被随机选择。数据文件的格式如下:

- 第一行:物品数量、颜色数量、箱子容量
- 对每个物品,给出其重量和颜色


## 模型

彩色装箱问题 (CBPP) 的 OptAgent 模型使用 [集合决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。对于每个箱子,我们定义一个集合变量来表示分配到该箱子的物品集合。我们将集合变量约束为形成一个 [partition](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#partition),以确保每个物品恰好属于一个箱子。

我们使用集合上的可变参 **sum** 算子和一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)(返回与任何物品索引相关联的重量)来计算箱子的总重量。注意,该求和中项的数量在搜索过程中是变化的,集合的大小也随之变化。然后我们可以将总重量约束为小于箱子容量。

我们还使用集合上的可变参 **sum** 算子和一个 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)(若物品具有给定颜色则返回 1,否则返回 0)来计算箱子中给定颜色的元素数量。该求和中项的数量在搜索过程中也是变化的,集合的大小也随之变化。然后我们可以将每个求和约束为小于 1。

如果一个箱子至少包含一个物品,则该箱子被实际使用。利用 **count** 算子(返回集合中元素的数量),我们可以检查每个箱子是否被实际使用,然后计算出所用箱子的总数。

模型计算最优箱子数量的简单上下界。它仅定义 nbMaxBins 个集合变量,并使用 [hxObjectiveThreshold](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/builtinfunctions.html#hxObjectiveThreshold) 在达到 nbMinBins 个箱子的解时停止搜索。


## Python 实现


In [ ]:
from pathlib import Path

from optagent import ModelBuilder, solve


def read_instance(filename):
    values = [int(value) for value in Path(filename).read_text(encoding="utf-8").split()]
    nb_items, nb_colors, bin_capacity = values[:3]
    weights = values[3::2][:nb_items]
    colors = values[4::2][:nb_items]
    return nb_items, nb_colors, bin_capacity, weights, colors


def main(instance_file, output_file=None, time_limit=5):
    nb_items, nb_colors, bin_capacity, weights_data, colors_data = read_instance(instance_file)
    model = ModelBuilder()
    bins = [model.set(nb_items, name=f"bin_{index}") for index in range(nb_items)]
    model.constraint(model.partition(bins), name="partition")
    weights = model.array(weights_data)
    colors = model.array(colors_data)

    bin_weights = []
    for bin_items in bins:
        weight = model.sum(bin_items, model.lambda_function(lambda item: weights[item]))
        model.constraint(weight <= bin_capacity)
        for color in range(nb_colors):
            count = model.sum(bin_items, model.lambda_function(lambda item, color=color: colors[item] == color))
            model.constraint(count <= 1)
        bin_weights.append(weight)

    bins_used = model.sum(*(bin_items.count() > 0 for bin_items in bins))
    model.minimize(bins_used, name="bins_used")
    solution = solve(model, time_limit_s=float(time_limit))
    values = solution.values(
        {
            "bins_used": bins_used,
            **{f"bin_{index}": bin_items for index, bin_items in enumerate(bins)},
            **{f"weight_{index}": weight for index, weight in enumerate(bin_weights)},
        }
    )
    lines = [f"Bins used = {values['bins_used']}; Status = {solution.status.value}"]
    for index in range(nb_items):
        items = values[f"bin_{index}"]
        if items:
            lines.append(f"Bin {index + 1}: weight={values[f'weight_{index}']}; items={' '.join(map(str, items))}")
    result = "\n".join(lines)
    print(result)
    if output_file is not None:
        Path(output_file).write_text(result + "\n", encoding="utf-8")
    return solution

## 运行实例


In [ ]:
INSTANCE_DIR = Path.cwd() / "instances"

In [ ]:
solution = main(INSTANCE_DIR / "201_2500_DI_0.txt-Q2", time_limit=5)